# LinguABC: Chatterbox Turbo GPU pilot (Kaggle / Colab, free T4)

Runs the LinguABC voice worker's real pipeline with the real Chatterbox Turbo model on a GPU and measures speed, VRAM and quality.

**Content is synthetic:** scripts written for this pilot, three synthetic reference voices (designed with Qwen3-TTS VoiceDesign, no real person). No LinguABC user data, no real answer keys, no secrets. Nothing here talks to LinguABC or Supabase.

1. **Kaggle:** Settings → Accelerator **GPU T4 x2**, Internet **On** (phone-verified account). Upload `linguabc-gpu-pilot.zip` as a Dataset and add it to the notebook.  
   **Colab:** Runtime → Change runtime type → **T4 GPU**. Run the upload cell.
2. Run all cells (about 20-40 minutes including installs). Download `linguabc-gpu-pilot-results.zip` at the end and send it back.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os, glob, zipfile, shutil
WORK = '/kaggle/working' if os.path.isdir('/kaggle') else '/content'
os.chdir(WORK)
DEST = os.path.join(WORK, 'linguabc-gpu-pilot')
if not os.path.isdir(DEST):
    # Kaggle unpacks an uploaded zip dataset; it may also keep the zip. Inputs are read-only, so copy.
    unpacked = glob.glob('/kaggle/input/**/linguabc-gpu-pilot/pilot/gpu/gpu_pilot.py', recursive=True)
    zips = glob.glob('/kaggle/input/**/linguabc-gpu-pilot.zip', recursive=True)
    if unpacked:
        shutil.copytree(unpacked[0].rsplit('/pilot/gpu/', 1)[0], DEST)
    elif zips:
        zipfile.ZipFile(zips[0]).extractall(WORK)
    else:
        from google.colab import files  # Colab: choose linguabc-gpu-pilot.zip
        files.upload()
        zipfile.ZipFile(os.path.join(WORK, 'linguabc-gpu-pilot.zip')).extractall(WORK)
os.chdir(DEST)
print(os.getcwd(), sorted(os.listdir('pilot/gpu/voices')))

In [ ]:
# Same pins as workers/tts/docker/chatterbox.Dockerfile (chatterbox-tts 0.1.7 pins torch 2.6.0, CUDA 12.4 wheels).
!pip -q install 'setuptools<81' 'chatterbox-tts==0.1.7' 'faster-whisper==1.2.1' resemblyzer soundfile 'speechbrain>=1.0' 2>&1 | tail -3
!which ffmpeg && ffmpeg -version | head -1

In [ ]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
# Accent screening model (MIT): a signal only; a person decides by listening.
from huggingface_hub import snapshot_download
ACCENT = snapshot_download('Jzuluaga/accent-id-commonaccent_ecapa')
print(ACCENT)

In [ ]:
# The benchmark. If faster-whisper cannot use the GPU (cuDNN), add --asr-device cpu and rerun.
!TQDM_DISABLE=1 python pilot/gpu/gpu_pilot.py --voices-dir pilot/gpu/voices --out results --cases short,section,expr --asr large-v3 --accent-dir "$ACCENT" 2>&1 | grep -v Warning | tail -40

In [ ]:
print(open('results/summary.md').read())

In [ ]:
shutil.make_archive(os.path.join(WORK, 'linguabc-gpu-pilot-results'), 'zip', 'results')
print('download:', os.path.join(WORK, 'linguabc-gpu-pilot-results.zip'))
try:
    from google.colab import files; files.download(os.path.join(WORK, 'linguabc-gpu-pilot-results.zip'))
except ImportError:
    pass  # Kaggle: Output tab -> linguabc-gpu-pilot-results.zip